# Khojki round four — completed and archived

**9 training runs completed**, including two 3,000-step extensions; five optical inference variants also evaluated. No candidate beat the retained ensemble on the unchanged development set (14/405 edits).

- Fresh page-21 test: **21/120 edits, 17.50% CER**; previous model identical. Five non-expert references were frozen before predictions.
- Found and fixed an RGB padding defect; its matched accuracy experiment did not show a gain.
- Source-based reference audit proposes changes that move the same older output from 26/123 to 15/128 edits. **That is a reference effect, not a model gain.**
- CPU/GPU page text matched. Two-thread CPU: **3.28 seconds for 16 lines**. 16 integration tests passed in Colab.
- **All 12 result archives were saved locally and SHA-256/ZIP verified.** Complete observed output text is preserved in `cloud/Khojki_round4_executed.ipynb`; full training histories and best/last optimizer/RNG checkpoints are saved in the project.
- Large embedded download payloads were removed from this notebook only after verified archival, to reduce browser memory. The code remains reproducible using the two input bundles.
- **Runtime deleted: 0 active sessions, 0 units/hour, 166.94 units remaining.**

Local project: `outputs/khojki-ocr`. See `RESULTS.md`, `technical-retro.pdf`, `results/round4-comparison.html`, and the archive receipts. No retraining is needed for inference.


In [ ]:
from pathlib import Path
import sys,os,json,time,zipfile,hashlib,subprocess,shutil,datetime
BUNDLE=Path('/content/khojki-round4.zip');assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()=='48a70d7957b90929f205d661f1f38cecb4c6f509c913ac0a35573f054bc2436e'
ROOT=Path('/content/khojki-ocr')
with zipfile.ZipFile(BUNDLE) as z:z.extractall('/content')
for n,h in json.loads((ROOT/'cloud-manifest.json').read_text()).items():assert hashlib.sha256((ROOT/n).read_bytes()).hexdigest()==h,n
subprocess.run([sys.executable,'-m','pip','install','-q','freetype-py','uharfbuzz','scipy','opencv-python-headless','pytest','pypdf','pypdfium2'],check=True,timeout=180)
os.environ['PYTHONPATH']=str(ROOT/'src')
import torch
assert torch.cuda.is_available();print(torch.cuda.get_device_name(0))
ANN=ROOT/'data/round2/train-dev.json';EXP=ROOT/'experiments';RES=ROOT/'results';RES.mkdir(exist_ok=True)
def run(args,log,timeout=900):
 with log.open('w') as f:p=subprocess.run([sys.executable,*map(str,args)],stdout=f,stderr=subprocess.STDOUT,timeout=timeout)
 print(log.name,log.read_text()[-1300:]);p.check_returncode()
run(['-m','pytest',ROOT/'tests','-q'],RES/'round4-tests.log',120)


In [ ]:
p=ROOT/'scripts/finetune.py'
s=p.read_text().replace('    if a.freeze_rnn:model.rnn.eval()\n','').replace('    for param in model.rnn.parameters():param.requires_grad_(False)','    for param in model.rnn.parameters():param.requires_grad_(False)\n    model.rnn.dropout=0.0')
p.write_text(s)
CASES={'r4-control':[],'r4-word-red':['--word-fraction','.4','--word-geometry','preserve','--legacy-rgb-word-padding'],'r4-word-white':['--word-fraction','.4','--word-geometry','preserve'],'r4-freezebn':['--freeze-bn'],'r4-word-white-freezebn':['--word-fraction','.4','--word-geometry','preserve','--freeze-bn'],'r4-freezernn':['--freeze-rnn']}
def train(name,steps):
 folder=EXP/name;folder.mkdir(exist_ok=True);local=folder/'last.pt';resume=local.exists();checkpoint=local if resume else EXP/'precision-packed/best.pt'
 if resume and torch.load(local,map_location='cpu',weights_only=False)['step']>=steps:print('Already complete',name,steps);return
 run([ROOT/'scripts/finetune.py','--checkpoint',checkpoint,'--annotations',ANN,'--steps',steps,'--lr','0.00005','--seed','503','--layout','robust','--bold-replay','--packed','--device','cuda','--output',name,*CASES[name],*(['--resume'] if resume else [])],EXP/(name+f'-{steps}.log'))
for name in CASES:train(name,600)
run([ROOT/'scripts/select_round4_model.py','--phase','pilot'],EXP/'round4-pilot.log')
report=json.loads((EXP/'round4-pilot-selection.json').read_text());eligible=sorted([r for r in report['experiments'] if r['name'] in CASES and r['errors']<=16],key=lambda r:r['errors'])[:2]
print('Extension rule: at most two pilots with <=16/405 edits; chosen',[r['name'] for r in eligible])
for row in eligible:train(row['name'],3000)
run([ROOT/'scripts/select_round4_model.py','--phase','final'],EXP/'round4-final.log')
# These already-viewed test sets are regression diagnostics, not new blind estimates.
for old in [2,3]:
 run([ROOT/'scripts/evaluate_pages.py','--checkpoint',ROOT/'models/round4-selected.pt','--annotations',ROOT/f'data/round{old}/test.json','--split','test','--device','cuda','--output',RES/f'round4-regression-r{old}'],RES/f'round4-regression-r{old}.log')


In [ ]:
from IPython.display import display,HTML
import base64
EXPORTS=[]
for name in [*CASES,'round4-selection']:
 target=Path('/content')/(name+'-results.zip')
 with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
  paths=list((EXP/name).glob('*'))+list(EXP.glob(name+'-*.log')) if name!='round4-selection' else list(RES.rglob('*'))+list(EXP.glob('round4-*'))+[ROOT/'models/round4-selected.pt']
  for p in paths:
   if p.is_file():z.write(p,str(p.relative_to(ROOT)))
 display(HTML('<a download="'+target.name+'" href="data:application/zip;base64,'+base64.b64encode(target.read_bytes()).decode()+'">Download '+name+' results</a>'))
 EXPORTS.append({'name':target.name,'bytes':target.stat().st_size,'sha256':hashlib.sha256(target.read_bytes()).hexdigest()})
print('EXPORT_MANIFEST',json.dumps(EXPORTS))


In [ ]:
# Offline-prepared follow-up: matched low learning rate and source-audited label correction.
bundle=Path('/content/khojki-round4-followup.zip')
assert hashlib.sha256(bundle.read_bytes()).hexdigest()=='df4c43f08a52be96bddfb5434e979f1cb3188dbf49005231da5be00047733fca'
with zipfile.ZipFile(bundle) as z:z.extractall(ROOT)
run([ROOT/'scripts/round4_followup.py'],EXP/'round4-followup.log',2400)
run([ROOT/'scripts/evaluate_round4_final.py'],RES/'round4-final-evaluation.log',900)


In [ ]:
EXPORTS_FOLLOWUP=[]
for name in ['r4-low-original','r4-low-corrected','r4-low-white','round4-followup']:
 target=Path('/content')/(name+'-results.zip')
 with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
  paths=list((EXP/name).glob('*'))+list(EXP.glob(name+'-*.log')) if name!='round4-followup' else list(RES.rglob('*'))+list(EXP.glob('round4-followup*'))+[ROOT/'models/round4-final.pt']
  for p in paths:
   if p.is_file():z.write(p,str(p.relative_to(ROOT)))
 display(HTML('<a download="'+target.name+'" href="data:application/zip;base64,'+base64.b64encode(target.read_bytes()).decode()+'">Download '+name+' results</a>'))
 EXPORTS_FOLLOWUP.append({'name':target.name,'bytes':target.stat().st_size,'sha256':hashlib.sha256(target.read_bytes()).hexdigest()})
print('FOLLOWUP_MANIFEST',json.dumps(EXPORTS_FOLLOWUP))


In [ ]:
# Final deployment parity and CPU throughput, run after model selection/evaluation.
out=RES/'round4-cpu-demo'
run(['-m','khojki_ocr',ROOT/'data/pages/p21.png','--checkpoint',ROOT/'models/round4-final.pt','--output',out,'--device','cpu'],RES/'round4-cpu.log',180)
cpu=json.loads((out/'page-001/result.json').read_text());gpu=json.loads((RES/'round4-final-r4/p21/result.json').read_text());assert [r['text'] for r in cpu['lines']]==[r['text'] for r in gpu['lines']]
saved=torch.load(ROOT/'models/round4-final.pt',map_location='cpu',weights_only=False)
members=saved.get('ensemble_states') or [{'model':saved['model'],'packed':saved.get('packed',False)}]
verify={'cpu_gpu_page_text_equal':True,'cpu_page':{k:v for k,v in cpu.items() if k!='lines'},'cpu_run':json.loads((out/'run.json').read_text()),'members':len(members),'parameters':sum(sum(v.numel() for k,v in m['model'].items() if not any(s in k for s in ['running_mean','running_var','num_batches_tracked'])) for m in members),'views':saved.get('views',[[1.,0.]]),'checkpoint_sha256':hashlib.sha256((ROOT/'models/round4-final.pt').read_bytes()).hexdigest()}
(RES/'round4-verification.json').write_text(json.dumps(verify,indent=2));print('VERIFICATION',json.dumps(verify))
target=Path('/content/round4-verification-results.zip')
with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
 for p in [*out.rglob('*'),RES/'round4-verification.json',RES/'round4-cpu.log']:
  if p.is_file():z.write(p,str(p.relative_to(ROOT)))
display(HTML('<a download="'+target.name+'" href="data:application/zip;base64,'+base64.b64encode(target.read_bytes()).decode()+'">Download round4 verification</a>'))
print('VERIFY_MANIFEST',json.dumps([{'name':target.name,'bytes':target.stat().st_size,'sha256':hashlib.sha256(target.read_bytes()).hexdigest()}]))
